# Todo #6 — 特征工程 FE-5（purgedcv WalkForwardSplit + OOF Autoencoder）

> 镜像 [`credit-fraud-feature-engineering-4.ipynb`](credit-fraud-feature-engineering-4.ipynb)（TimeSeriesSplit + AE）；**验证差异：** `purgedcv.WalkForwardSplit` + **embargo 2h**。

**目标：** 降 FP（1 EUR 优先）> 降 FN；AUC-PR ablation 定稿 `MODEL_FEATURES_V2`。
**导出：** `MODEL_FEATURES_V2_purgedcv.json`


## 0.1 FE-4 vs FE-3：purgedcv 验证差异

| 维度 | FE-3 | FE-4（本 notebook） |
|------|------|---------------------|
| CV 切分器 | `sklearn.TimeSeriesSplit` | `purgedcv.WalkForwardSplit`（扩展窗口） |
| Embargo | 无 | `CV_EMBARGO = 2h`（连续 `Time` 秒轴，非 `hours_since_start` 桶） |
| Purge | 无 | `CV_PURGE_HORIZON = 0`（`Class` 瞬时标签，无重叠预测视界） |
| 折间自检 | 无 | `assert_no_temporal_leakage` 每折断言 |
| hold-out 切分 | 纯时间比例 | `temporal_train_val_split(use_embargo=True)` 裁 train 尾 |

**为何加 embargo？** EDA 欺诈率按时模式显示：早峰 **3–7** 时（自 2 时起抬升）、谷段 **8–24** 时、晚峰 **25–28** 时（**26** 时最高）。相邻时段交易特征相关，纯时间切分易让 train 尾渗透 val 头；在连续 `Time` 秒上预留 **2 小时**缓冲更贴近部署时的信息边界。

**镜像关系：** 特征工程流程与 [`credit-fraud-feature-engineering-3.ipynb`](credit-fraud-feature-engineering-3.ipynb) 一致；仅验证策略升级为 purged walk-forward。


## 0. 环境与数据加载


In [4]:
# --- 0. 环境与数据加载 ---
# 镜像主 notebook；读入 creditcard.csv 并记录全局欺诈率。
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 40)



def resolve_creditcard_path() -> Path:
    """自 cwd 向上查找项目根下的 input/creditcard.csv（兼容 src/ 与 src/feature-engineering/）。"""
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / 'input' / 'creditcard.csv'
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        '未找到 input/creditcard.csv；请在项目根或 src/feature-engineering 下运行 notebook。'
    )


DATA_PATH = resolve_creditcard_path()


def read_creditcard_csv(path) -> pd.DataFrame:
    """依次尝试 utf-8 / 容错 utf-8 / latin-1，避免 UnicodeDecodeError。"""
    for kwargs in (
        {'encoding': 'utf-8'},
        {'encoding': 'utf-8', 'encoding_errors': 'replace'},
        {'encoding': 'latin-1'},
    ):
        try:
            return pd.read_csv(path, **kwargs)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError('utf-8', b'', 0, 1, 'failed to decode creditcard.csv')


df = read_creditcard_csv(DATA_PATH)
FRAUD_RATE = df['Class'].mean()
print(f'行数: {len(df):,}  |  欺诈笔数: {df["Class"].sum()}  |  欺诈率: {FRAUD_RATE:.4f}')


行数: 284,807  |  欺诈笔数: 492  |  欺诈率: 0.0017


## 1. 建模工具（FE-4 purgedcv 验证版）

- **§1.1** 特征与分类器（EDA、LGB/XGB、单次 hold-out 评估）
- **§1.2** purgedcv WalkForwardSplit CV（时间切分、embargo、交叉验证）


In [5]:
# --- 1.1 特征与分类器（EDA + LGB/XGB 训练评估）---
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score, f1_score,
    precision_score, recall_score, confusion_matrix,
)
import lightgbm as lgb
import xgboost as xgb

V_COLS = [c for c in df.columns if c.startswith('V')]
BASE_FEATURES = V_COLS + ['Amount', 'Time']

EDA_FEATURES = [
    'log1p_amount',
    'hours_since_start',
    'is_micro_testing',
    'is_one_euro',
    'is_amount_1_30',    # (1, 30] EUR — 难样本/PCA 易混带
    'is_amount_75_110',  # [75, 110] EUR — 难样本/PCA 易混带
]

EDA_FEATURE_DOC = {
    'log1p_amount': 'log1p(Amount)',
    'hours_since_start': 'Time // 3600',
    'is_micro_testing': 'Amount < 1',
    'is_one_euro': 'Amount == 1',
    'is_amount_1_30': '1 < Amount <= 30',
    'is_amount_75_110': '75 <= Amount <= 110',
}
AMOUNT_BAND_FEATURES = ('is_amount_1_30', 'is_amount_75_110')

EARLY_STOPPING_ROUNDS = 50
MAX_BOOST_ROUNDS = 1500
ES_FRAC = 0.25  # 早停监控集占当前 train 折比例（75% fit / 25% ES）
DEFAULT_CLASSIFICATION_THRESHOLD = 0.5
TOP_V_K = 6  # Family A/B 门控交叉用的 Top-V 个数
CV_N_SPLITS = 5           # 组合矩阵 / ablation 可选 CV 折数
CV_RANDOM_STATE = 42


def build_eda_features(data: pd.DataFrame) -> pd.DataFrame:
    """构造 EDA 衍生特征（难样本金额带版；无 is_inactive / is_small_testing）。"""
    out = data.copy()
    out['log1p_amount'] = np.log1p(out['Amount'])
    out['hours_since_start'] = (out['Time'] // 3600).astype(int)
    out['is_micro_testing'] = out['Amount'] < 1
    out['is_one_euro'] = out['Amount'] == 1.0
    out['is_amount_1_30'] = (out['Amount'] > 1) & (out['Amount'] <= 30)
    out['is_amount_75_110'] = (out['Amount'] >= 75) & (out['Amount'] <= 110)
    return out


def _split_early_stop_set(X_tr, y_tr, es_frac=ES_FRAC, random_state=42):
    """切分出用来验证早停的数据"""
    return train_test_split(
        X_tr, y_tr, test_size=es_frac, random_state=random_state, stratify=y_tr,
    )


def make_classifier(model_name: str, y_train: pd.Series, params: dict | None = None):
    """构造树（logloss + 类别不平衡权重）"""
    params = dict(params or {})
    spw = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
    if model_name == 'LightGBM':
        defaults = dict(
            n_estimators=MAX_BOOST_ROUNDS, learning_rate=0.05, max_depth=6, num_leaves=31,
            min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1,
            class_weight='balanced', random_state=42, verbose=-1,
        )
        defaults.update(params)
        return lgb.LGBMClassifier(**defaults)
    defaults = dict(
        n_estimators=MAX_BOOST_ROUNDS, learning_rate=0.05, max_depth=6,
        min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, scale_pos_weight=spw,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        random_state=42, eval_metric='logloss', verbosity=0,
    )
    defaults.update(params)
    defaults['early_stopping_rounds'] = EARLY_STOPPING_ROUNDS
    return xgb.XGBClassifier(**defaults)


def fit_classifier(clf, model_name: str, X_tr, y_tr, X_es=None, y_es=None):
    """训练（logloss + 早停）"""
    if X_es is None or y_es is None:
        clf.fit(X_tr, y_tr)
        return clf
    if model_name == 'LightGBM':
        clf.fit(
            X_tr, y_tr, eval_set=[(X_es, y_es)], eval_metric='binary_logloss',
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
        )
    else:
        clf.fit(X_tr, y_tr, eval_set=[(X_es, y_es)], verbose=False)
    return clf


def eval_classifier(
    model_name: str, X_train, y_train, X_val, y_val,
    threshold: float = DEFAULT_CLASSIFICATION_THRESHOLD, random_state: int = 42,
) -> dict:
    """不交叉验证，直接训练评估模型"""
    X_fit, X_es, y_fit, y_es = _split_early_stop_set(X_train, y_train, random_state=random_state)
    clf = make_classifier(model_name, y_fit)
    fit_classifier(clf, model_name, X_fit, y_fit, X_es, y_es)
    proba = clf.predict_proba(X_val)[:, 1]
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    return {
        '模型': model_name, '特征数': X_train.shape[1],
        'AUC-PR': average_precision_score(y_val, proba),
        f'F1@{threshold}': f1_score(y_val, pred, zero_division=0),
        f'Precision@{threshold}': precision_score(y_val, pred, zero_division=0),
        f'Recall@{threshold}': recall_score(y_val, pred, zero_division=0),
        'FP': fp, 'FN': fn, 'proba': proba, 'pred': pred, 'threshold': threshold,
    }


df_model = build_eda_features(df)
print('衍生特征已构造；ablation 基线 BASE_FEATURES：')
print(BASE_FEATURES)
print('\nEDA 列（难样本金额带版）：')
for col in EDA_FEATURES:
    print(f"  {col:20s}  {EDA_FEATURE_DOC[col]}")
for col in AMOUNT_BAND_FEATURES:
    n = int(df_model[col].sum())
    print(f"  → {col}: {n:,} 笔 ({n / len(df_model):.2%})")


衍生特征已构造；ablation 基线 BASE_FEATURES：
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Time']

EDA 列（难样本金额带版）：
  log1p_amount          log1p(Amount)
  hours_since_start     Time // 3600
  is_micro_testing      Amount < 1
  is_one_euro           Amount == 1
  is_amount_1_30        1 < Amount <= 30
  is_amount_75_110      75 <= Amount <= 110
  → is_amount_1_30: 130,588 笔 (45.85%)
  → is_amount_75_110: 20,464 笔 (7.19%)
=== purgedcv WalkForwardSplit 折诊断 ===
CV_EMBARGO=0 days 02:00:00 | CV_PURGE_HORIZON=0 days 00:00:00
EDA 欺诈率时序：早峰 3–7h ↑、谷 8–24h、晚峰 25–28h（26h 最高）→ embargo 在连续秒轴上隔离相邻时段


,折,train 行数,val 行数,embargo 裁掉 train 尾,train Time 末,val Time 起,val Time 末,val hour 范围,覆盖早峰3-7,覆盖晚峰25-28,val 欺诈率
0,1,47472,47467,0,43221s,43222s,65098s,12–18,False,False,0.0015
1,2,94939,47467,0,65098s,65099s,84694s,18–23,False,False,0.0011
2,3,142406,47467,0,84694s,84695s,128593s,23–35,False,True,0.0021
3,4,189868,47467,5,128592s,128593s,149198s,35–41,False,False,0.0013
4,5,237338,47467,2,149197s,149198s,172792s,41–47,False,False,0.0013


## 1.2 purgedcv WalkForwardSplit CV

连续 `Time` 秒轴上的 walk-forward 切分 + **embargo 2h** + 折内 `assert_no_temporal_leakage` 自检。
下游 §6.4 组合矩阵、`cross_val_eval`、`feature_ablation`（hold-out）均依赖本节函数。


In [ ]:
# --- 1.2 purgedcv WalkForwardSplit CV（时间序列切分 + embargo + 交叉验证）---
from purgedcv import WalkForwardSplit  # apply_embargo 由 WalkForwardSplit 折内自动调用
from purgedcv.diagnostics import assert_no_temporal_leakage

CV_EMBARGO = pd.Timedelta(hours=2)       # 连续 Time 秒轴上的折后缓冲（非 hours 桶化）
CV_PURGE_HORIZON = pd.Timedelta(0)       # Class 为瞬时标签，无重叠预测视界

# 全局绑定表：iter_purged_cv_folds(len(X)) 未显式传 data 时回退此处
_CV_BOUND_DATA: pd.DataFrame | None = None


def sort_by_time(data: pd.DataFrame) -> pd.DataFrame:
    """按 Time 升序排序并重置索引，供时间序列验证使用。"""
    return data.sort_values('Time', kind='mergesort').reset_index(drop=True)


def bind_cv_data(data: pd.DataFrame) -> pd.DataFrame:
    """按 Time 升序排序并绑定全局 _CV_BOUND_DATA，供 iter_purged_cv_folds(len(X)) 使用。"""
    global _CV_BOUND_DATA
    out = sort_by_time(data)
    _CV_BOUND_DATA = out
    return out


def build_cv_timestamps(data: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    """
    从连续 Time（秒）构造 purgedcv 所需的 prediction_times / evaluation_times。

    信用卡欺诈标签 Class 为瞬时观测（交易当下即知真伪），故 prediction_time == evaluation_time。
    使用 pd.to_timedelta(Time, unit='s') 保持连续秒级时间轴——**不做** hours_since_start 桶化。
    """
    t = pd.to_timedelta(data['Time'].astype(float), unit='s')
    return t.copy(), t.copy()


def make_walk_forward_cv(
    n_samples: int,
    n_splits: int = CV_N_SPLITS,
    data: pd.DataFrame | None = None,
) -> WalkForwardSplit:
    """
    构造 purgedcv WalkForwardSplit（扩展窗口 + embargo + purge）。

    - test_size 折宽与 FE-3 一致：n // (n_splits + 1)
    - window='expanding'：每折 train 取 test 之前的全部历史（walk-forward 部署语义）
    - embargo=CV_EMBARGO（2h）：基于 EDA 欺诈率时序图——早峰 3–7 时（自 2 时起抬升）、
      谷段 8–24 时、晚峰 25–28 时（26 时最高）；在连续 Time 秒上预留缓冲，防相邻时段渗透
    - purge_horizon=0：Class 瞬时标签，训练行标签视界不与 test 重叠
    """
    bound = data if data is not None else _CV_BOUND_DATA
    if bound is None:
        raise RuntimeError('先调用 bind_cv_data() 绑定按 Time 排序的数据表')
    pred, evalu = build_cv_timestamps(bound)
    test_size = max(1, n_samples // (n_splits + 1))
    return WalkForwardSplit(
        n_splits=n_splits,
        test_size=test_size,
        window='expanding',
        prediction_times=pred,
        evaluation_times=evalu,
        purge_horizon=CV_PURGE_HORIZON,
        embargo=CV_EMBARGO,
    )


def iter_purged_cv_folds(
    n_samples: int | None = None,
    n_splits: int = CV_N_SPLITS,
    data: pd.DataFrame | None = None,
):
    """
    purgedcv WalkForwardSplit 折迭代器：yield (train_idx, val_idx)。

    兼容 FE-3 的 iter_purged_cv_folds(len(X)) 调用方式：未传 data 时使用 _CV_BOUND_DATA。
    每折 yield 前由 splitter 内部执行 purge + embargo；yield 后 assert_no_temporal_leakage 自检。
    """
    bound = data if data is not None else _CV_BOUND_DATA
    if bound is None:
        raise RuntimeError('先调用 bind_cv_data() 绑定按 Time 排序的数据表')
    n = n_samples if n_samples is not None else len(bound)
    cv = make_walk_forward_cv(n, n_splits=n_splits, data=bound)
    pred, evalu = build_cv_timestamps(bound)
    for tr_idx, va_idx in cv.split(np.arange(n)):
        assert_no_temporal_leakage(
            tr_idx, va_idx,
            prediction_times=pred,
            evaluation_times=evalu,
            purge_horizon=CV_PURGE_HORIZON,
        )
        yield tr_idx, va_idx


def describe_purged_cv_folds(
    data: pd.DataFrame | None = None,
    n_splits: int = CV_N_SPLITS,
) -> pd.DataFrame:
    """
    诊断各 purged walk-forward 折：规模、Time 范围、embargo 裁剪与欺诈率语境。

    EDA 欺诈率按 hours_since_start 的模式（解读 embargo=2h 的业务依据，**非**切分桶）：
      - 早峰 3–7 时（自 2 时起抬升）
      - 谷段 8–24 时
      - 晚峰 25–28 时（26 时欺诈率最高）
    embargo 在连续 Time 秒轴上裁掉 val 折结束后 CV_EMBARGO 内的 train 尾行。
    purge_horizon=0：瞬时 Class 标签，无标签视界重叠需 purge。
    """
    bound = data if data is not None else _CV_BOUND_DATA
    if bound is None:
        raise RuntimeError('先调用 bind_cv_data() 绑定按 Time 排序的数据表')
    n = len(bound)
    rows = []
    for fold, (tr_idx, va_idx) in enumerate(iter_purged_cv_folds(n_samples=n, n_splits=n_splits, data=bound), start=1):
        raw_train_end = int(va_idx.min())
        raw_train_n = raw_train_end
        embargo_dropped = raw_train_n - len(tr_idx)
        tr_time = bound['Time'].iloc[tr_idx]
        va_time = bound['Time'].iloc[va_idx]
        va_fraud = float(bound['Class'].iloc[va_idx].mean())
        va_hours = bound['hours_since_start'].iloc[va_idx] if 'hours_since_start' in bound.columns else (va_time // 3600).astype(int)
        h_min, h_max = int(va_hours.min()), int(va_hours.max())
        hrs = set(range(h_min, h_max + 1))
        hit_early = bool(hrs & set(range(3, 8)))   # EDA 早峰 3–7（hour 2 起抬升，诊断用 3–7）
        hit_late = bool(hrs & set(range(25, 29)))  # EDA 晚峰 25–28（26 最高）
        rows.append({
            '折': fold,
            'train 行数': len(tr_idx),
            'val 行数': len(va_idx),
            'embargo 裁掉 train 尾': embargo_dropped,
            'train Time 末': f"{tr_time.max():.0f}s",
            'val Time 起': f"{va_time.min():.0f}s",
            'val Time 末': f"{va_time.max():.0f}s",
            'val hour 范围': f'{h_min}–{h_max}',
            '覆盖早峰3-7': hit_early,
            '覆盖晚峰25-28': hit_late,
            'val 欺诈率': f'{va_fraud:.4f}',
        })
    df_diag = pd.DataFrame(rows)
    print('=== purgedcv WalkForwardSplit 折诊断 ===')
    print(f'CV_EMBARGO={CV_EMBARGO} | CV_PURGE_HORIZON={CV_PURGE_HORIZON}')
    print('EDA 欺诈率时序：早峰 3–7h ↑、谷 8–24h、晚峰 25–28h（26h 最高）→ embargo 在连续秒轴上隔离相邻时段')
    return df_diag


def _trim_train_pre_test_embargo(
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    pred: pd.Series,
    embargo: pd.Timedelta,
) -> np.ndarray:
    """
    hold-out / walk-forward 训练集：剔除验证起点前 embargo 窗口内的 train 行。

    purgedcv.apply_embargo 面向「测试段之后仍有训练行」的 PurgedKFold；
    本 notebook 的 hold-out 只有过去训练，故在验证起点前留 CV_EMBARGO 空档。
    """
    if embargo <= pd.Timedelta(0) or len(val_idx) == 0 or len(train_idx) == 0:
        return train_idx
    val_start = pred.iloc[val_idx].min()
    keep = pred.iloc[train_idx] < (val_start - embargo)
    return train_idx[keep.to_numpy()]


def temporal_train_val_split(
    data: pd.DataFrame,
    y_col: str = 'Class',
    test_size: float = 0.2,
    use_embargo: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """
    时间切分：前 (1-test_size) 训练，后 test_size 验证（已按 Time 排序）。

    use_embargo=True 时，训练尾部剔除验证起点前 CV_EMBARGO（2h 连续秒），
    与 WalkForwardSplit 折内 embargo 语义一致（避免紧邻验证段的 train 渗透）。
    """
    data = sort_by_time(data)
    n = len(data)
    split_at = int(n * (1 - test_size))
    train_idx = np.arange(split_at, dtype=int)
    val_idx = np.arange(split_at, n, dtype=int)
    if use_embargo and CV_EMBARGO > pd.Timedelta(0):
        pred, _ = build_cv_timestamps(data)
        train_idx = _trim_train_pre_test_embargo(train_idx, val_idx, pred, CV_EMBARGO)
    train = data.iloc[train_idx]
    val = data.iloc[val_idx]
    return train, val, train[y_col], val[y_col]

def cross_val_auc_pr(
    model_name: str,
    X: pd.DataFrame,
    y: pd.Series,
    n_splits: int = CV_N_SPLITS,
    random_state: int = CV_RANDOM_STATE,
) -> tuple[float, float, list[float]]:
    """purgedcv WalkForwardSplit：返回 (AUC-PR 均值, 标准差, 各折分数列表)。"""
    fold_scores: list[float] = []
    for tr_idx, va_idx in iter_purged_cv_folds(len(X), n_splits=n_splits):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        X_fit, X_es, y_fit, y_es = _split_early_stop_set(X_tr, y_tr, random_state=random_state)
        clf = make_classifier(model_name, y_fit)
        fit_classifier(clf, model_name, X_fit, y_fit, X_es, y_es)
        fold_scores.append(float(average_precision_score(y_va, clf.predict_proba(X_va)[:, 1])))
    arr = np.array(fold_scores)
    return float(arr.mean()), float(arr.std(ddof=0)), fold_scores


def get_oof_proba(
    model_name: str,
    X: pd.DataFrame,
    y: pd.Series,
    n_splits: int = CV_N_SPLITS,
    random_state: int = CV_RANDOM_STATE,
) -> np.ndarray:
    """purgedcv WalkForwardSplit OOF 概率，用于 CV 下汇总 FP/FN（@threshold）。"""
    oof = np.zeros(len(y), dtype=float)
    for tr_idx, va_idx in iter_purged_cv_folds(len(X), n_splits=n_splits):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_fit, X_es, y_fit, y_es = _split_early_stop_set(X_tr, y_tr, random_state=random_state)
        clf = make_classifier(model_name, y_fit)
        fit_classifier(clf, model_name, X_fit, y_fit, X_es, y_es)
        oof[va_idx] = clf.predict_proba(X.iloc[va_idx])[:, 1]
    return oof


def cross_val_eval(
    model_name: str,
    data: pd.DataFrame,
    feature_cols: list,
    y_col: str = 'Class',
    n_splits: int = CV_N_SPLITS,
    random_state: int = CV_RANDOM_STATE,
    threshold: float = DEFAULT_CLASSIFICATION_THRESHOLD,
) -> dict:
    """CV 评估：一轮 purged walk-forward 同时算各折 AUC-PR 与 OOF 概率（避免重复训练）。"""
    X = data[feature_cols]
    y = data[y_col]
    oof_proba = np.zeros(len(y), dtype=float)
    fold_scores: list[float] = []

    for tr_idx, va_idx in iter_purged_cv_folds(len(X), n_splits=n_splits):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        X_fit, X_es, y_fit, y_es = _split_early_stop_set(X_tr, y_tr, random_state=random_state)
        clf = make_classifier(model_name, y_fit)
        fit_classifier(clf, model_name, X_fit, y_fit, X_es, y_es)
        proba_va = clf.predict_proba(X_va)[:, 1]
        fold_scores.append(float(average_precision_score(y_va, proba_va)))
        oof_proba[va_idx] = proba_va

    arr = np.array(fold_scores)
    pred = (oof_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    return {
        '模型': model_name,
        '特征数': len(feature_cols),
        'AUC-PR_mean': float(arr.mean()),
        'AUC-PR_std': float(arr.std(ddof=0)),
        'fold_AUC-PR': fold_scores,
        f'F1@{threshold}': f1_score(y, pred, zero_division=0),
        f'Precision@{threshold}': precision_score(y, pred, zero_division=0),
        f'Recall@{threshold}': recall_score(y, pred, zero_division=0),
        'FP': int(fp),
        'FN': int(fn),
    }


def feature_ablation(
    data: pd.DataFrame, candidate_features=None, base_features=None,
    model_name: str = 'LightGBM', test_size: float = 0.2, random_state: int = 42,
    include_all_combo: bool = True,
    threshold: float = DEFAULT_CLASSIFICATION_THRESHOLD,
) -> pd.DataFrame:
    """特征消融；并列输出 FP / FN（@threshold）。"""
    if base_features is None:
        base_features = BASE_FEATURES
    candidate_features = list(candidate_features or [])
    data = sort_by_time(data)
    train, val, y_train, y_val = temporal_train_val_split(data, test_size=test_size)
    specs = [('基线', base_features)]
    for f in candidate_features:
        specs.append((f'+{f}', base_features + [f]))
    if include_all_combo and candidate_features:
        specs.append(('+全部候选', base_features + candidate_features))
    rows, base_ap = [], None
    for label, cols in specs:
        missing = [c for c in cols if c not in data.columns]
        if missing:
            raise KeyError(f'缺少列: {missing}')
        res = eval_classifier(
            model_name, train[cols], y_train,
            val[cols], y_val, threshold=threshold,
        )
        if base_ap is None:
            base_ap = res['AUC-PR']
        rows.append({
            '特征集': label, '特征数': res['特征数'], 'AUC-PR': res['AUC-PR'],
            f"F1@{threshold}": res[f'F1@{threshold}'],
            f"Precision@{threshold}": res[f'Precision@{threshold}'],
            f"Recall@{threshold}": res[f'Recall@{threshold}'],
            'FP': res['FP'], 'FN': res['FN'],
            'Δ AUC-PR': res['AUC-PR'] - base_ap,
        })
    return pd.DataFrame(rows).sort_values('AUC-PR', ascending=False).reset_index(drop=True)


df_model = bind_cv_data(df_model)
display(describe_purged_cv_folds(df_model))



## 6.0 基础设施：Top-V 选择、门控交叉、OOF Autoencoder


In [6]:
# =============================================================================
# 6.0 特征工程基础设施
# -----------------------------------------------------------------------------
# 本 cell 提供三类能力：
#   1. pick_top_v_features  — 用树模型 importance 选出与欺诈最相关的 Top-K 个 V 列
#   2. build_cross_features — 构造「？ × V」交叉特征（Family A/B 共用）
#   3. oof_ae_reconstruction_error — OOF Autoencoder，产出无泄露的异常分 ae_oof_error
# =============================================================================

# --- 依赖导入 ---
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras

# --- Autoencoder 超参数（可按训练时间 / 效果微调）---
AE_INPUT_COLS = V_COLS          # AE 只吃 V1–V28（PCA 分量），不含 Amount/Time（Amount 强偏态）
AE_LATENT_DIM = 8               # 瓶颈层维度：压缩正常流形，欺诈/异常更难重构
AE_MAX_EPOCHS = 40              # 每折最大 epoch；实际由 EarlyStopping 提前结束
AE_BATCH_SIZE = 256             # mini-batch 大小
AE_PATIENCE = 5                 # val_loss 连续 5 epoch 不降则停训
AE_MAX_NORMAL_SAMPLES = 50_000  # 每折 normal 子采样上限（全表 normal≈28 万，全训太慢）
AE_RANDOM_STATE = 42            # 控制 purged walk-forward 折划分、子采样、TF 随机种子


def pick_top_v_features(
    data: pd.DataFrame,
    feature_cols: list | None = None,
    k: int = TOP_V_K,
    model_name: str = 'LightGBM',
    random_state: int = 42,
) -> list:
    """
    在指定特征集上训一版树分类器，按 feature_importances_（gain）取 Top-K 个 V 列。

      - feature_cols: list | None  → Python 3.10+ 联合类型；None 时用 BASE_FEATURES

    为何需要：Family A 交叉项是 is_one_euro × V_i，试图解决FP误报，不能对 28 个 V 全交叉（维数爆炸），
    先用模型 importance 筛出与欺诈最相关的 V，再与 is_one_euro 相乘。
    """
    feature_cols = feature_cols or BASE_FEATURES  # `or`：None / 空列表时回退默认
    data = sort_by_time(data)
    train, val, y_train, y_val = temporal_train_val_split(data, test_size=0.2)
    X_train = train[feature_cols]
    X_val = val[feature_cols]

    clf = make_classifier(model_name, y_train)
    # 再从 X_train 切 ES_FRAC（25%）作早停监控集（与 2.0 eval_classifier 一致）
    X_fit, X_es, y_fit, y_es = _split_early_stop_set(X_train, y_train, random_state=random_state)
    fit_classifier(clf, model_name, X_fit, y_fit, X_es, y_es)

    # feature_importances_ 与 feature_cols 一一对应；只保留 V 列并降序
    imp = pd.Series(clf.feature_importances_, index=feature_cols)
    v_imp = imp[V_COLS].sort_values(ascending=False)
    top_v = list(v_imp.head(k).index)  # .head(k) 取前 k 个 index，转 list
    print(f'Top-{k} V（{model_name} gain）：{top_v}')
    return top_v


def build_cross_features(
    data: pd.DataFrame,
    top_v: list,
    gate_col: str = 'is_one_euro',
    prefix: str = 'one_euro',
) -> tuple[pd.DataFrame, list]:
    """
    特征交叉：new_col = gate_col × V_i。

      - 返回 tuple[pd.DataFrame, list]：新 DataFrame + 新列名列表（供 ablation 传入）

    业务含义：one_euro_Vi 等同于告诉树：「这一维 Vi 的信息，请只在 1 EUR 子空间里用。」
    帮助树模型区分「1 EUR 正常探测」与「1 EUR 欺诈」在 V 空间上的差异（降 FP）。
    否则一棵树更可能在全表上学：Vi > 某阈值 → 更像欺诈
    这个规则会作用在 所有金额 上，包括大量非 1 EUR 交易
    在 1 EUR 上，正常探测和欺诈的 Vi 可能重叠很大，全局阈值就容易误伤正常 1 EUR 交易（FP）
    """
    out = data.copy()            
    gate = out[gate_col].astype(float)  # bool → 0.0/1.0，便于与连续 V 相乘
    new_cols = []
    for v in top_v:
        name = f'{prefix}_{v}'  # 如 one_euro_V14
        out[name] = gate * out[v]   # 逐元素相乘（Series × Series）
        new_cols.append(name)
    return out, new_cols


def _build_ae(input_dim: int, latent_dim: int = AE_LATENT_DIM) -> keras.Model:
    """
    全连接 Autoencoder：input_dim → 16 → latent → 16 → input_dim。

    Keras Functional API：
      - keras.Input(shape=(input_dim,))：定义输入张量形状 (batch, input_dim)
      - 每层 Dense(...)(x)：函数式 API，x 是上一层的输出
      - 最后一层 Dense(input_dim) 无激活：重构原始维度，配合 MSE loss
      - keras.Model(inp, out)：指定输入/输出节点，构成可训练模型

    中间层 ReLU；瓶颈 latent 迫使模型学正常样本的低维表示（？）。
    """
    inp = keras.Input(shape=(input_dim,))
    x = keras.layers.Dense(16, activation='relu')(inp)
    x = keras.layers.Dense(latent_dim, activation='relu')(x)
    x = keras.layers.Dense(16, activation='relu')(x)
    out = keras.layers.Dense(input_dim)(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='mse')
    return model


def oof_ae_reconstruction_error(
    data: pd.DataFrame,
    feature_cols: list | None = None,
    y_col: str = 'Class',
    n_splits: int = 5,
    random_state: int = AE_RANDOM_STATE,
) -> np.ndarray:
    """
    Out-of-Fold Autoencoder 重构误差（样本级 MSE，越大越像异常）。

    核心防泄露设计：
      - 每折 AE 只在 train 折的 **正常样本** 上训练（学「正常流形」）
      - 重构误差只算 **val 折** 样本 → 该样本从未参与该折 AE 的训练
      - 禁止用 in-sample 误差当特征（否则欺诈/正常在训练集上的误差会偏乐观）

    返回值：长度 = len(data) 的 np.ndarray，与 data 行序对齐，可直接 df['ae_oof_error'] = ...
    """
    feature_cols = feature_cols or AE_INPUT_COLS
    X = data[feature_cols].values.astype(np.float32) 
    y = data[y_col].values
    oof_err = np.zeros(len(y), dtype=np.float32)      # 预分配空间，按 va_idx 填入各折结果

    for fold, (tr_idx, va_idx) in enumerate(
        iter_purged_cv_folds(data=data, n_splits=n_splits), start=1,
    ):
        normal_tr = tr_idx[y[tr_idx] == 0]            # 训练 AE 时剔除欺诈，只学 normal

        # 子采样：normal 过多时随机抽 AE_MAX_NORMAL_SAMPLES 条，加速且通常足够
        if len(normal_tr) > AE_MAX_NORMAL_SAMPLES:
            rng = np.random.default_rng(random_state + fold)  # 每折不同种子，可复现
            normal_tr = rng.choice(normal_tr, size=AE_MAX_NORMAL_SAMPLES, replace=False)

        X_normal = X[normal_tr]
        scaler = StandardScaler()
        X_normal_s = scaler.fit_transform(X_normal)     # 只在 normal 上 fit μ、σ
        X_va_s = scaler.transform(X[va_idx])          # val 用同一 scaler transform（含欺诈）

        tf.keras.backend.clear_session()                # 每折新建模型前清图，防内存累积
        tf.random.set_seed(random_state + fold)
        ae = _build_ae(X.shape[1])

        es = keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=AE_PATIENCE, restore_best_weights=True, verbose=0,
        )
        # 从 normal 训练集末尾切 10% 作 AE 内部 val（仅用于早停，不是 OOF val）
        n_val = max(1, int(0.1 * len(X_normal_s)))
        ae.fit(
            X_normal_s[:-n_val], X_normal_s[:-n_val],   # 输入=目标（自编码器）
            validation_data=(X_normal_s[-n_val:], X_normal_s[-n_val:]),
            epochs=AE_MAX_EPOCHS, batch_size=AE_BATCH_SIZE,
            callbacks=[es], verbose=0,
        )

        recon = ae.predict(X_va_s, verbose=0)           # shape: (n_val_fold, n_features)
        # axis=1：对每个样本的所有特征维求 MSE 均值 → 标量异常分
        oof_err[va_idx] = np.mean((X_va_s - recon) ** 2, axis=1)
        print(f'  fold {fold}/{n_splits} 完成（normal 训练样本 {len(normal_tr):,}）')
    return oof_err


# --- 执行：在 BASE 特征上跑一版 LGB，得到 Family A 要交叉的 Top-6 V ---
TOP_V = pick_top_v_features(df_model, BASE_FEATURES, k=TOP_V_K, model_name='LightGBM')


Top-6 V（LightGBM gain）：['V14', 'V4', 'V12', 'V7', 'V26', 'V10']


## 6.1 Family A — 1 EUR 特征交叉（FP 优先）

`one_euro_{Vi} = is_one_euro × V_i`，仅在 1 EUR 区激活对应主成分信号。


In [7]:
from IPython.display import display

# =============================================================================
# 6.1 Family A — 1 EUR 门控交叉 + ablation
# -----------------------------------------------------------------------------
# 流程：构造交叉列 → 对 LightGBM / XGBoost 分别做 feature_ablation
# ablation 基线默认为 BASE_FEATURES（V + Amount + Time），见 §1 的 feature_ablation
# =============================================================================

# build_cross_features 返回：
#   df_fe          — 在 df_model 上追加 one_euro_V* 列后的完整表
#   CROSS_FAMILY_A — 新列名列表，如 ['one_euro_V14', 'one_euro_V4', ...]
df_fe, CROSS_FAMILY_A = build_cross_features(
    df_model, TOP_V, gate_col='is_one_euro', prefix='one_euro',
)

print(f'Family A 共 {len(CROSS_FAMILY_A)} 列：')
# 展示「列名 → 定义」对照表，便于报告引用
display(pd.DataFrame({'特征': CROSS_FAMILY_A, '定义': [f'is_one_euro × {v}' for v in TOP_V]}))

# --- LightGBM ablation ---
# include_all_combo=True：除「基线」「+单列」外，再跑一行「+全部候选」（6 列一起加）
# 6.4 定稿用「+全部候选」行的 Δ AUC-PR 判断是否整族入选
print('\n=== Family A ablation（LightGBM，基线=BASE_FEATURES）===')
family_a_lgb = feature_ablation(
    df_fe, candidate_features=CROSS_FAMILY_A,
    model_name='LightGBM', include_all_combo=True,
)
display(family_a_lgb.round(4))  # round(4) 便于阅读；Δ AUC-PR 相对基线行

# --- XGBoost ablation（同一划分、同一候选，对比两模型是否一致受益）---
print('\n=== Family A ablation（XGBoost）===')
family_a_xgb = feature_ablation(
    df_fe, candidate_features=CROSS_FAMILY_A,
    model_name='XGBoost', include_all_combo=True,
)
display(family_a_xgb.round(4))


Family A 共 6 列：


,特征,定义
0,one_euro_V14,is_one_euro × V14
1,one_euro_V4,is_one_euro × V4
2,one_euro_V12,is_one_euro × V12
3,one_euro_V7,is_one_euro × V7
4,one_euro_V26,is_one_euro × V26
5,one_euro_V10,is_one_euro × V10



=== Family A ablation（LightGBM，基线=BASE_FEATURES）===


,特征集,特征数,AUC-PR,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR
0,基线,30,0.8023,0.8261,0.9048,0.7600,6,18,0.0000
1,+one_euro_V4,31,0.8022,0.8382,0.9344,0.7600,4,18,-0.0001
2,+one_euro_V7,31,0.7997,0.8296,0.9333,0.7467,4,19,-0.0026
3,+全部候选,36,0.7994,0.8235,0.9180,0.7467,5,19,-0.0029
4,+one_euro_V14,31,0.7975,0.8382,0.9344,0.7600,4,18,-0.0048
5,+one_euro_V12,31,0.7973,0.8235,0.9180,0.7467,5,19,-0.0050
6,+one_euro_V26,31,0.7964,0.8296,0.9333,0.7467,4,19,-0.0059
7,+one_euro_V10,31,0.7960,0.8209,0.9322,0.7333,4,20,-0.0063



=== Family A ablation（XGBoost）===


,特征集,特征数,AUC-PR,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR
0,+全部候选,36,0.8079,0.8296,0.9333,0.7467,4,19,0.0066
1,+one_euro_V7,31,0.8029,0.8201,0.8906,0.7600,7,18,0.0016
2,+one_euro_V14,31,0.8024,0.8116,0.8889,0.7467,7,19,0.0011
3,基线,30,0.8013,0.8175,0.9032,0.7467,6,19,0.0000
4,+one_euro_V4,31,0.7995,0.8201,0.8906,0.7600,7,18,-0.0019
5,+one_euro_V10,31,0.7994,0.8201,0.8906,0.7600,7,18,-0.0020
6,+one_euro_V26,31,0.7989,0.8261,0.9048,0.7600,6,18,-0.0024
7,+one_euro_V12,31,0.7979,0.8116,0.8889,0.7467,7,19,-0.0034


## 6.2 Family B（占位）+ Family C 说明

**Family B（待做）：** `band_{Vi} = is_amount_1_30 × V_i` 或 `is_amount_75_110 × V_i`，针对难样本金额带内 FN。
镜像 6.1，将 `gate_col='is_amount_1_30'`（或 `is_amount_75_110`）、`prefix='band_low'` 即可。

**Family C（本轮不做）：** 可选 `log1p_amount × is_amount_1_30`、`log1p_amount × V_top` 等可解释连续门控。


In [8]:
# =============================================================================
# 6.2 Family B — 占位（尚未实现）
# -----------------------------------------------------------------------------
# 设计：与 6.1 镜像，门控改为 is_amount_1_30 / is_amount_75_110，针对难样本金额带 FN。
# 取消注释下方代码即可跑通；TOP_V 可复用 6.0 结果，也可在 BASE 上重新 pick_top_v。
# =============================================================================

# TODO: 待做 — 镜像 6.1
# df_fe, CROSS_FAMILY_B = build_cross_features(
#     df_fe, TOP_V, gate_col='is_amount_1_30', prefix='band_low',
# )
# family_b_lgb = feature_ablation(
#     df_fe, candidate_features=CROSS_FAMILY_B,
#     model_name='LightGBM', include_all_combo=True,
# )

print('Family B 尚未实现；见上方 markdown 说明。')


Family B 尚未实现；见上方 markdown 说明。


## 6.3 OOF Autoencoder 重构误差

purgedcv WalkForwardSplit 每折 **仅用正常样本** 训练 AE；折外 MSE 作为 `ae_oof_error`（越大越偏离正常流形）。
与树模型 **不同数学原理**，非树模型 `p_fraud`，无目标泄露。


In [9]:
from IPython.display import display

# =============================================================================
# 6.3 OOF Autoencoder — 计算 ae_oof_error 并 ablation
# -----------------------------------------------------------------------------
# ae_oof_error：与树模型不同原理的异常分，可作为 BASE 上的增量特征。
# 注意：全表 5-fold OOF 较慢（每折训一个 AE），首次运行需数分钟。
# =============================================================================

print('计算 OOF AE 重构误差（purgedcv WalkForwardSplit 5-fold，可能需数分钟）…')
# 返回值长度 = len(df_fe)，按 index 对齐写入新列
df_fe['ae_oof_error'] = oof_ae_reconstruction_error(df_fe, feature_cols=AE_INPUT_COLS)
print(f"ae_oof_error: mean={df_fe['ae_oof_error'].mean():.6f}, "
      f"std={df_fe['ae_oof_error'].std():.6f}")

# --- 单特征 ablation：只加 ae_oof_error 一列 ---
# include_all_combo=False：候选只有 1 列时无需「+全部候选」行
print('\n=== OOF AE ablation（LightGBM）===')
ae_ablation_lgb = feature_ablation(
    df_fe, candidate_features=['ae_oof_error'],
    model_name='LightGBM', include_all_combo=False,
)
display(ae_ablation_lgb.round(4))

print('\n=== OOF AE ablation（XGBoost）===')
ae_ablation_xgb = feature_ablation(
    df_fe, candidate_features=['ae_oof_error'],
    model_name='XGBoost', include_all_combo=False,
)
display(ae_ablation_xgb.round(4))

# --- 可选：AE 误差 × 1 EUR 门控 ---
# 假设：异常分在 1 EUR 子空间更有判别力；乘 is_one_euro 后非 1 EUR 行为 0
# base_features 显式设为 BASE + ae_oof_error，测「在已有 AE 分上再加门控交叉」的边际收益
df_fe['ae_oof_error_x_one_euro'] = df_fe['ae_oof_error'] * df_fe['is_one_euro'].astype(float)
print('\n=== ae_oof_error × is_one_euro ablation（LightGBM）===')
ae_gate_ablation = feature_ablation(
    df_fe, base_features=BASE_FEATURES + ['ae_oof_error'],
    candidate_features=['ae_oof_error_x_one_euro'],
    model_name='LightGBM', include_all_combo=False,
)
display(ae_gate_ablation.round(4))


计算 OOF AE 重构误差（purgedcv WalkForwardSplit 5-fold，可能需数分钟）…
  fold 1/5 完成（normal 训练样本 47,326）
  fold 2/5 完成（normal 训练样本 50,000）
  fold 3/5 完成（normal 训练样本 50,000）
  fold 4/5 完成（normal 训练样本 50,000）
  fold 5/5 完成（normal 训练样本 50,000）
ae_oof_error: mean=0.456203, std=3.370509

=== OOF AE ablation（LightGBM）===


,特征集,特征数,AUC-PR,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR
0,基线,30,0.8023,0.8261,0.9048,0.7600,6,18,0.000
1,+ae_oof_error,31,0.7953,0.8296,0.9333,0.7467,4,19,-0.007



=== OOF AE ablation（XGBoost）===


,特征集,特征数,AUC-PR,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR
0,基线,30,0.8013,0.8175,0.9032,0.7467,6,19,0.0000
1,+ae_oof_error,31,0.7993,0.8058,0.8750,0.7467,8,19,-0.0021



=== ae_oof_error × is_one_euro ablation（LightGBM）===


,特征集,特征数,AUC-PR,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR
0,+ae_oof_error_x_one_euro,32,0.8006,0.8235,0.9180,0.7467,5,19,0.0053
1,基线,31,0.7953,0.8296,0.9333,0.7467,4,19,0.0000


## 6.4 组合验证 + 定稿 MODEL_FEATURES_V2（FE-4 purgedcv 验证版）

**6.4a**（purgedcv WalkForwardSplit 5-fold CV + embargo，LGB / XGB **分表**）覆盖 **全部人为特征** 及组合：

| 块 | 内容 |
|----|------|
| **EDA** | `log1p_amount`、`hours_since_start`、`is_micro_testing`、`is_one_euro`、`is_amount_1_30`（(1,30]）、`is_amount_75_110`（[75,110]）；无 `is_inactive` / `is_small_testing` |
| **Family A** | `one_euro×Top-V`（单列 / Top-2/3 / 全族） |
| **OOF AE** | `ae_oof_error`、`ae_oof_error×is_one_euro` |
| **跨块** | EDA+AE、EDA+Family A、EDA+AE+Family A、**全部人为特征** |

**6.4b**：双模型 CV Δ 均为正且 Δ_mean 最高 → 定稿 `MODEL_FEATURES_V2`。


In [10]:
from IPython.display import display

# =============================================================================
# 6.4a 全部人为特征组合矩阵（EDA + Family A + OOF AE；purgedcv WalkForwardSplit CV；LGB/XGB 分表）
# =============================================================================

FE_EDA = list(EDA_FEATURES)  # log1p, hours, micro, one_euro, 难样本金额带 (1,30]/[75,110]
FE_AE = ['ae_oof_error']
FE_AE_GATE = ['ae_oof_error_x_one_euro']
FE_FAMILY_A = list(CROSS_FAMILY_A)
EDA_CURATED = ['hours_since_start', 'is_one_euro']  # 主 notebook 2.1d curated


def _dedupe_preserve_order(cols: list) -> list:
    seen, out = set(), []
    for c in cols:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out


def _extra_summary(extra_cols: list) -> str:
    if not extra_cols:
        return '—'
    s = ', '.join(extra_cols)
    return s if len(s) <= 72 else s[:69] + '...'


def _dedupe_specs(specs: list[tuple[list, str]]) -> list[tuple[list, str]]:
    seen: set[tuple[str, ...]] = set()
    out: list[tuple[list, str]] = []
    for extra, label in specs:
        key = tuple(_dedupe_preserve_order(extra))
        if key in seen:
            continue
        seen.add(key)
        out.append((list(key), label))
    return out


def build_fe_combo_specs(
    family_a_cols: list,
    eda_cols: list | None = None,
    fe_ae: list | None = None,
    fe_ae_gate: list | None = None,
    top_k_subsets: tuple[int, ...] = (2, 3),
) -> list[tuple[list, str]]:
    """
    构造 **全部人为特征** 的组合规格（相对 BASE_FEATURES 的增量列）。
    块：EDA 1.7 → OOF AE → Family A → 跨块并集。
    """
    eda_cols = list(eda_cols or FE_EDA)
    fe_ae = list(fe_ae or FE_AE)
    fe_ae_gate = list(fe_ae_gate or FE_AE_GATE)
    specs: list[tuple[list, str]] = []

    eda_minus_bands = [c for c in eda_cols if c not in AMOUNT_BAND_FEATURES]
    eda_minus_no_hours = [c for c in eda_minus_bands if c != 'hours_since_start']
    eda_minus_no_hours_log = [c for c in eda_minus_no_hours if c != 'log1p_amount']

    # --- 0. 基线 ---
    specs.append(([], '0. 基线（仅 BASE）'))

    # --- EDA：单列 ---
    for i, col in enumerate(eda_cols, start=1):
        specs.append(([col], f'Ed{i}. BASE + {col}'))

    # --- EDA：2.1d 式组合（镜像主 notebook）---
    specs.append((eda_cols, 'Ed_all. BASE + 全部 EDA（6 列）'))
    specs.append((eda_minus_bands, 'Ed_noBands. BASE + EDA 去掉难样本金额带'))
    specs.append((list(AMOUNT_BAND_FEATURES), 'Ed_bands. BASE + 两档难样本金额带'))
    specs.append((eda_minus_no_hours, 'Ed_noHours. BASE + EDA 再去掉 hours'))
    specs.append((eda_minus_no_hours_log, 'Ed_noHoursLog. BASE + EDA 再去掉 log1p'))
    specs.append((EDA_CURATED, 'Ed_cur. BASE + hours + is_one_euro'))

    # --- OOF AE ---
    specs.append((fe_ae, 'AE. BASE + ae_oof_error'))
    specs.append((fe_ae + fe_ae_gate, 'AE+gate. BASE + ae_oof_error + ae×1EUR'))

    # --- EDA × AE ---
    specs.append((eda_cols + fe_ae, 'Ed_all+AE. 全部 EDA + ae_oof_error'))
    specs.append((EDA_CURATED + fe_ae, 'Ed_cur+AE. hours+is_one_euro + ae_oof_error'))
    specs.append((eda_minus_no_hours_log + fe_ae, 'Ed_noHoursLog+AE. EDA(无金额带/hours/log1p) + AE'))
    for col in eda_cols:
        specs.append((fe_ae + [col], f'AE+{col}. ae_oof_error + {col}'))

    # --- Family A：单列 / +AE / Top-k / 全族 ---
    for i, col in enumerate(family_a_cols, start=1):
        short = col.replace('one_euro_', '')
        specs.append(([col], f'A{i}. BASE + {col}（{short}）'))
        specs.append((fe_ae + [col], f'A{i}+AE. BASE + ae + {col}（{short}）'))
    for k in top_k_subsets:
        if k >= len(family_a_cols):
            continue
        subset = family_a_cols[:k]
        v_short = '+'.join(c.replace('one_euro_', '') for c in subset)
        specs.append((subset, f'A_top{k}. Family A Top-{k}（{v_short}）'))
        specs.append((fe_ae + subset, f'A_top{k}+AE. ae + Family A Top-{k}'))
    specs.append((family_a_cols, 'A_all. Family A 全族（6 列）'))
    specs.append((fe_ae + family_a_cols, 'A_all+AE. ae + Family A 全族'))

    # --- EDA × Family A（Top-k，不加 AE）---
    for k in top_k_subsets:
        if k >= len(family_a_cols):
            continue
        subset = family_a_cols[:k]
        specs.append((eda_cols + subset, f'Ed_all+A_top{k}. 全部 EDA + Family A Top-{k}'))
        specs.append((EDA_CURATED + subset, f'Ed_cur+A_top{k}. curated EDA + Family A Top-{k}'))

    # --- EDA × AE × Family A ---
    for k in top_k_subsets:
        if k >= len(family_a_cols):
            continue
        subset = family_a_cols[:k]
        specs.append((eda_cols + fe_ae + subset, f'Ed_all+AE+A_top{k}. 全部 EDA + ae + Family A Top-{k}'))
        specs.append((EDA_CURATED + fe_ae + subset, f'Ed_cur+AE+A_top{k}. curated + ae + Family A Top-{k}'))

    # --- 全量人为特征（EDA + AE + Family A + AE 门控）---
    all_handcrafted = _dedupe_preserve_order(eda_cols + fe_ae + family_a_cols + fe_ae_gate)
    specs.append((all_handcrafted, 'FULL. BASE + 全部人为特征（EDA+AE+FamilyA+gate）'))


    # --- Ed_bands × Family A / AE（并入主矩阵）---
    _bands = list(AMOUNT_BAND_FEATURES)
    for k in top_k_subsets:
        if k >= len(family_a_cols):
            continue
        subset = family_a_cols[:k]
        specs.append((_bands + subset, f'Ed_bands+A_top{k}. 金额带 + Family A Top-{k}'))
        specs.append((_bands + fe_ae + subset, f'Ed_bands+AE+A_top{k}. 金额带 + ae + Family A Top-{k}'))

    return _dedupe_specs(specs)


def eval_fe_combo(
    data: pd.DataFrame,
    extra_cols: list,
    label: str,
    model_name: str = 'LightGBM',
    n_splits: int = CV_N_SPLITS,
    random_state: int = CV_RANDOM_STATE,
    threshold: float = DEFAULT_CLASSIFICATION_THRESHOLD,
) -> dict:
    """purgedcv WalkForwardSplit CV 评估 BASE + extra_cols（比单次 hold-out 更稳）。"""
    extra_cols = _dedupe_preserve_order(list(extra_cols))
    feature_cols = _dedupe_preserve_order(BASE_FEATURES + extra_cols)
    missing = [c for c in feature_cols if c not in data.columns]
    if missing:
        raise KeyError(f'[{label}] 缺少列: {missing}')

    res = cross_val_eval(
        model_name, data, feature_cols,
        n_splits=n_splits, random_state=random_state, threshold=threshold,
    )
    return {
        '特征组合': label,
        '增量列数': len(extra_cols),
        '总特征数': len(feature_cols),
        '增量摘要': _extra_summary(extra_cols),
        '增量列': extra_cols,
        'AUC-PR_mean': res['AUC-PR_mean'],
        'AUC-PR_std': res['AUC-PR_std'],
        f'F1@{threshold}': res[f'F1@{threshold}'],
        f'Precision@{threshold}': res[f'Precision@{threshold}'],
        f'Recall@{threshold}': res[f'Recall@{threshold}'],
        'FP': res['FP'],
        'FN': res['FN'],
    }


def run_fe_combo_matrix(
    data: pd.DataFrame,
    combo_specs: list[tuple[list, str]],
    model_name: str = 'LightGBM',
    verbose: bool = True,
) -> pd.DataFrame:
    rows = []
    n = len(combo_specs)
    for i, (extra, label) in enumerate(combo_specs, start=1):
        if verbose:
            print(f'  [{model_name}] CV {i}/{n}: {label}')
        rows.append(eval_fe_combo(data, extra, label, model_name=model_name))
    df_out = pd.DataFrame(rows)
    base_ap = float(df_out.loc[df_out['特征组合'].str.startswith('0.'), 'AUC-PR_mean'].iloc[0])
    df_out['Δ AUC-PR vs BASE'] = df_out['AUC-PR_mean'] - base_ap
    return df_out.sort_values('AUC-PR_mean', ascending=False).reset_index(drop=True)


def _display_combo_table(df: pd.DataFrame, title: str) -> None:
    """展示组合表（隐藏增量列 list，用增量摘要代替）。"""
    show_cols = [c for c in df.columns if c != '增量列']
    print(title)
    display(df[show_cols].round(4))


df_fe = bind_cv_data(sort_by_time(df_fe))

COMBO_SPECS = build_fe_combo_specs(FE_FAMILY_A)
_n_eda = len(FE_EDA)
print(f'共 {len(COMBO_SPECS)} 种组合 × {CV_N_SPLITS}-fold purgedcv WalkForwardSplit CV')
print(f'  含 EDA 1.7（{_n_eda} 列单列+组合）、Family A、OOF AE、跨块 FULL 等\n')

print('=== 6.4a LightGBM 组合矩阵（CV，按 AUC-PR_mean 降序）===')
combo_lgb = run_fe_combo_matrix(df_fe, COMBO_SPECS, model_name='LightGBM')
_display_combo_table(combo_lgb, '')

print(f'\n=== 6.4a XGBoost 组合矩阵（CV，按 AUC-PR_mean 降序）===')
combo_xgb = run_fe_combo_matrix(df_fe, COMBO_SPECS, model_name='XGBoost')
_display_combo_table(combo_xgb, '')

print('\n--- LightGBM：CV Δ>0 的前 3 名 ---')
display(combo_lgb.loc[combo_lgb['Δ AUC-PR vs BASE'] > 0].head(3).drop(columns=['增量列']).round(4))

print('--- XGBoost：CV Δ>0 的前 3 名 ---')
display(combo_xgb.loc[combo_xgb['Δ AUC-PR vs BASE'] > 0].head(3).drop(columns=['增量列']).round(4))

# 供 6.4b 定稿：内部合并，不在此 cell 展示宽表
combo_lgb_idx = combo_lgb.set_index('特征组合')
combo_xgb_idx = combo_xgb.set_index('特征组合')
_common_labels = combo_lgb_idx.index.intersection(combo_xgb_idx.index)
combo_dual_rows = []
for label in _common_labels:
    dl = float(combo_lgb_idx.loc[label, 'Δ AUC-PR vs BASE'])
    dx = float(combo_xgb_idx.loc[label, 'Δ AUC-PR vs BASE'])
    combo_dual_rows.append({
        '特征组合': label,
        '增量列': combo_lgb_idx.loc[label, '增量列'],
        'Δ_LGB': dl,
        'Δ_XGB': dx,
        'Δ_mean': (dl + dx) / 2,
        '双模型Δ均为正': dl > 0 and dx > 0,
    })
combo_dual = pd.DataFrame(combo_dual_rows).sort_values('Δ_mean', ascending=False).reset_index(drop=True)


共 55 种组合 × 5-fold purgedcv WalkForwardSplit CV
  含 EDA 1.7（6 列单列+组合）、Family A、OOF AE、跨块 FULL 等

=== 6.4a LightGBM 组合矩阵（CV，按 AUC-PR_mean 降序）===
  [LightGBM] CV 1/55: 0. 基线（仅 BASE）
  [LightGBM] CV 2/55: Ed1. BASE + log1p_amount
  [LightGBM] CV 3/55: Ed2. BASE + hours_since_start
  [LightGBM] CV 4/55: Ed3. BASE + is_micro_testing
  [LightGBM] CV 5/55: Ed4. BASE + is_one_euro
  [LightGBM] CV 6/55: Ed5. BASE + is_amount_1_30
  [LightGBM] CV 7/55: Ed6. BASE + is_amount_75_110
  [LightGBM] CV 8/55: Ed_all. BASE + 全部 EDA（6 列）
  [LightGBM] CV 9/55: Ed_noBands. BASE + EDA 去掉难样本金额带
  [LightGBM] CV 10/55: Ed_bands. BASE + 两档难样本金额带
  [LightGBM] CV 11/55: Ed_noHours. BASE + EDA 再去掉 hours
  [LightGBM] CV 12/55: Ed_noHoursLog. BASE + EDA 再去掉 log1p
  [LightGBM] CV 13/55: Ed_cur. BASE + hours + is_one_euro
  [LightGBM] CV 14/55: AE. BASE + ae_oof_error
  [LightGBM] CV 15/55: AE+gate. BASE + ae_oof_error + ae×1EUR
  [LightGBM] CV 16/55: Ed_all+AE. 全部 EDA + ae_oof_error
  [LightGBM] CV 17/55: Ed_cur+AE. h

,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE
0,A1. BASE + one_euro_V14（V14）,1,31,one_euro_V14,0.7865,0.0191,0.6542,0.8479,0.5325,47,230,0.0052
1,Ed_all+AE. 全部 EDA + ae_oof_error,7,37,"log1p_amount, hours_since_start, is_micro_test...",0.7857,0.0309,0.6525,0.8474,0.5305,47,231,0.0044
2,A2. BASE + one_euro_V4（V4）,1,31,one_euro_V4,0.7848,0.0242,0.6574,0.8642,0.5305,41,231,0.0035
3,Ed_bands. BASE + 两档难样本金额带,2,32,"is_amount_1_30, is_amount_75_110",0.7846,0.0371,0.6591,0.8595,0.5346,43,229,0.0033
4,A5+AE. BASE + ae + one_euro_V26（V26）,2,32,"ae_oof_error, one_euro_V26",0.7843,0.0229,0.6533,0.8502,0.5305,46,231,0.0030
5,Ed_all+A_top2. 全部 EDA + Family A Top-2,8,38,"log1p_amount, hours_since_start, is_micro_test...",0.7841,0.0262,0.6502,0.8297,0.5346,54,229,0.0028
6,Ed_cur. BASE + hours + is_one_euro,2,32,"hours_since_start, is_one_euro",0.7837,0.0358,0.6566,0.8667,0.5285,40,232,0.0024
7,Ed_noHours. BASE + EDA 再去掉 hours,3,33,"log1p_amount, is_micro_testing, is_one_euro",0.7836,0.0282,0.6600,0.8623,0.5346,42,229,0.0023
8,Ed_noHoursLog. BASE + EDA 再去掉 log1p,2,32,"is_micro_testing, is_one_euro",0.7833,0.0266,0.6515,0.8600,0.5244,42,234,0.0020
9,AE+hours_since_start. ae_oof_error + hours_sin...,2,32,"ae_oof_error, hours_since_start",0.7829,0.0355,0.6608,0.8651,0.5346,41,229,0.0016



=== 6.4a XGBoost 组合矩阵（CV，按 AUC-PR_mean 降序）===
  [XGBoost] CV 1/55: 0. 基线（仅 BASE）
  [XGBoost] CV 2/55: Ed1. BASE + log1p_amount
  [XGBoost] CV 3/55: Ed2. BASE + hours_since_start
  [XGBoost] CV 4/55: Ed3. BASE + is_micro_testing
  [XGBoost] CV 5/55: Ed4. BASE + is_one_euro
  [XGBoost] CV 6/55: Ed5. BASE + is_amount_1_30
  [XGBoost] CV 7/55: Ed6. BASE + is_amount_75_110
  [XGBoost] CV 8/55: Ed_all. BASE + 全部 EDA（6 列）
  [XGBoost] CV 9/55: Ed_noBands. BASE + EDA 去掉难样本金额带
  [XGBoost] CV 10/55: Ed_bands. BASE + 两档难样本金额带
  [XGBoost] CV 11/55: Ed_noHours. BASE + EDA 再去掉 hours
  [XGBoost] CV 12/55: Ed_noHoursLog. BASE + EDA 再去掉 log1p
  [XGBoost] CV 13/55: Ed_cur. BASE + hours + is_one_euro
  [XGBoost] CV 14/55: AE. BASE + ae_oof_error
  [XGBoost] CV 15/55: AE+gate. BASE + ae_oof_error + ae×1EUR
  [XGBoost] CV 16/55: Ed_all+AE. 全部 EDA + ae_oof_error
  [XGBoost] CV 17/55: Ed_cur+AE. hours+is_one_euro + ae_oof_error
  [XGBoost] CV 18/55: Ed_noHoursLog+AE. EDA(无金额带/hours/log1p) + AE
  [XGBoost] CV

,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE
0,A2. BASE + one_euro_V4（V4）,1,31,one_euro_V4,0.7833,0.0326,0.6616,0.8680,0.5346,40,229,0.0047
1,Ed_cur+AE+A_top2. curated + ae + Family A Top-2,5,35,"hours_since_start, is_one_euro, ae_oof_error, ...",0.7833,0.0337,0.6625,0.8656,0.5366,41,228,0.0046
2,Ed4. BASE + is_one_euro,1,31,is_one_euro,0.7829,0.0330,0.6567,0.8511,0.5346,46,229,0.0042
3,AE+gate. BASE + ae_oof_error + ae×1EUR,2,32,"ae_oof_error, ae_oof_error_x_one_euro",0.7827,0.0284,0.6658,0.8717,0.5386,39,227,0.0040
4,AE+is_one_euro. ae_oof_error + is_one_euro,2,32,"ae_oof_error, is_one_euro",0.7820,0.0297,0.6667,0.8800,0.5366,36,228,0.0033
5,A5. BASE + one_euro_V26（V26）,1,31,one_euro_V26,0.7820,0.0337,0.6574,0.8642,0.5305,41,231,0.0033
6,A1. BASE + one_euro_V14（V14）,1,31,one_euro_V14,0.7817,0.0364,0.6608,0.8651,0.5346,41,229,0.0030
7,Ed_bands+A_top3. 金额带 + Family A Top-3,5,35,"is_amount_1_30, is_amount_75_110, one_euro_V14...",0.7814,0.0318,0.6633,0.8792,0.5325,36,230,0.0027
8,Ed_all. BASE + 全部 EDA（6 列）,6,36,"log1p_amount, hours_since_start, is_micro_test...",0.7812,0.0331,0.6574,0.8642,0.5305,41,231,0.0025
9,Ed5. BASE + is_amount_1_30,1,31,is_amount_1_30,0.7810,0.0320,0.6600,0.8571,0.5366,44,228,0.0023



--- LightGBM：CV Δ>0 的前 3 名 ---


,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE
0,A1. BASE + one_euro_V14（V14）,1,31,one_euro_V14,0.7865,0.0191,0.6542,0.8479,0.5325,47,230,0.0052
1,Ed_all+AE. 全部 EDA + ae_oof_error,7,37,"log1p_amount, hours_since_start, is_micro_test...",0.7857,0.0309,0.6525,0.8474,0.5305,47,231,0.0044
2,A2. BASE + one_euro_V4（V4）,1,31,one_euro_V4,0.7848,0.0242,0.6574,0.8642,0.5305,41,231,0.0035


--- XGBoost：CV Δ>0 的前 3 名 ---


,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE
0,A2. BASE + one_euro_V4（V4）,1,31,one_euro_V4,0.7833,0.0326,0.6616,0.8680,0.5346,40,229,0.0047
1,Ed_cur+AE+A_top2. curated + ae + Family A Top-2,5,35,"hours_since_start, is_one_euro, ae_oof_error, ...",0.7833,0.0337,0.6625,0.8656,0.5366,41,228,0.0046
2,Ed4. BASE + is_one_euro,1,31,is_one_euro,0.7829,0.0330,0.6567,0.8511,0.5346,46,229,0.0042


定稿后可在 `Ed_bands`（两档金额带）、`Ed_cur+A_top3`、`Ed_noBands` 等候选间，用 **1 EUR 子集 FP** 做最终业务验收。

In [12]:
# =============================================================================
# 6.4b 定稿 MODEL_FEATURES_V2
# -----------------------------------------------------------------------------
# 展示：LGB / XGB 各自最优（Δ>0）；定稿：双模型 Δ 均为正且 Δ_mean 最高（combo_dual）
# =============================================================================

def _ablation_combo_delta(ablation_df: pd.DataFrame) -> float:
    row = ablation_df.loc[ablation_df['特征集'] == '+全部候选']
    if row.empty:
        row = ablation_df.iloc[[0]]
    return float(row['Δ AUC-PR'].iloc[0])


def _ablation_single_delta(ablation_df: pd.DataFrame, tag: str) -> float:
    row = ablation_df.loc[ablation_df['特征集'] == tag]
    return float(row['Δ AUC-PR'].iloc[0]) if not row.empty else 0.0


def _best_positive_row(combo_df: pd.DataFrame) -> pd.Series | None:
    pos = combo_df.loc[combo_df['Δ AUC-PR vs BASE'] > 0]
    return pos.iloc[0] if not pos.empty else None


decisions = []

# --- 各模型单独最优（仅展示，不一定作最终定稿）---
best_lgb = _best_positive_row(combo_lgb)
best_xgb = _best_positive_row(combo_xgb)

print('=== 6.4b LightGBM 单模型最优（CV Δ>0）===')
if best_lgb is not None:
    print(f"  {best_lgb['特征组合']}  |  Δ={best_lgb['Δ AUC-PR vs BASE']:.4f}  "
          f"|  AUC-PR={best_lgb['AUC-PR_mean']:.4f}±{best_lgb['AUC-PR_std']:.4f}  |  {best_lgb['增量摘要']}")
else:
    print('  无 CV Δ>0 组合')

print('\n=== 6.4b XGBoost 单模型最优（CV Δ>0）===')
if best_xgb is not None:
    print(f"  {best_xgb['特征组合']}  |  Δ={best_xgb['Δ AUC-PR vs BASE']:.4f}  "
          f"|  AUC-PR={best_xgb['AUC-PR_mean']:.4f}±{best_xgb['AUC-PR_std']:.4f}  |  {best_xgb['增量摘要']}")
else:
    print('  无 CV Δ>0 组合')

# --- 双模型一致定稿 ---
eligible = combo_dual[combo_dual['双模型Δ均为正']].sort_values('Δ_mean', ascending=False)
if eligible.empty:
    WINNER_LABEL = '0. 基线（仅 BASE）'
    SELECTED_EXTRA = []
    decisions.append('组合定稿：无「双模型 CV Δ 均为正」方案 → 保留 BASE_FEATURES')
else:
    winner = eligible.iloc[0]
    WINNER_LABEL = winner['特征组合']
    SELECTED_EXTRA = list(winner['增量列'])
    decisions.append(
        f"组合定稿：{WINNER_LABEL} "
        f"(LGB Δ={winner['Δ_LGB']:.4f}, XGB Δ={winner['Δ_XGB']:.4f}, Δ_mean={winner['Δ_mean']:.4f})"
    )

MODEL_FEATURES_V2 = BASE_FEATURES + [c for c in SELECTED_EXTRA if c not in BASE_FEATURES]

decisions.append(
    f'[参考] Family A 单族 ablation: LGB Δ={_ablation_combo_delta(family_a_lgb):.4f}, '
    f'XGB Δ={_ablation_combo_delta(family_a_xgb):.4f}'
)
decisions.append(
    f'[参考] ae_oof_error 单列 ablation: LGB Δ={_ablation_single_delta(ae_ablation_lgb, "+ae_oof_error"):.4f}, '
    f'XGB Δ={_ablation_single_delta(ae_ablation_xgb, "+ae_oof_error"):.4f}'
)

print('\n=== 6.4b 定稿决策 ===')
for d in decisions:
    print(' -', d)
print(f'\nMODEL_FEATURES_V2（{len(MODEL_FEATURES_V2)} 列，相对 BASE 增量 {len(MODEL_FEATURES_V2) - len(BASE_FEATURES)}）：')
print(MODEL_FEATURES_V2)

print('\n提示：若 LGB 与 XGB 单模型最优组合不同，可优先看业务指标（如 1 EUR FP）再手动微调。')

import json as _json
export_path = 'MODEL_FEATURES_V2_purgedcv.json'
with open(export_path, 'w', encoding='utf-8') as f:
    _json.dump(
        {
            'MODEL_FEATURES_V2': MODEL_FEATURES_V2,
            'winner_combo': WINNER_LABEL,
            'selected_extra': SELECTED_EXTRA,
            'best_lgb_combo': None if best_lgb is None else best_lgb['特征组合'],
            'best_xgb_combo': None if best_xgb is None else best_xgb['特征组合'],
            'cv_method': 'purgedcv.WalkForwardSplit',
            'cv_embargo': str(CV_EMBARGO),
            'cv_purge_horizon': str(CV_PURGE_HORIZON),
            'cv_n_splits': CV_N_SPLITS,
            'es_frac': ES_FRAC,
            'eda_features': EDA_FEATURES,
            'eda_feature_doc': EDA_FEATURE_DOC,
            'decisions': decisions,
            'combo_lgb': combo_lgb.drop(columns=['增量列']).round(6).to_dict(orient='records'),
            'combo_xgb': combo_xgb.drop(columns=['增量列']).round(6).to_dict(orient='records'),
        },
        f, ensure_ascii=False, indent=2,
    )
print(f'已导出 → {export_path}')


=== 6.4b LightGBM 单模型最优（CV Δ>0）===
  A1. BASE + one_euro_V14（V14）  |  Δ=0.0052  |  AUC-PR=0.7865±0.0191  |  one_euro_V14

=== 6.4b XGBoost 单模型最优（CV Δ>0）===
  A2. BASE + one_euro_V4（V4）  |  Δ=0.0047  |  AUC-PR=0.7833±0.0326  |  one_euro_V4

=== 6.4b 定稿决策 ===
 - 组合定稿：A1. BASE + one_euro_V14（V14） (LGB Δ=0.0052, XGB Δ=0.0030, Δ_mean=0.0041)
 - [参考] Family A 单族 ablation: LGB Δ=-0.0029, XGB Δ=0.0066
 - [参考] ae_oof_error 单列 ablation: LGB Δ=-0.0070, XGB Δ=-0.0021

MODEL_FEATURES_V2（31 列，相对 BASE 增量 1）：
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Time', 'one_euro_V14']

提示：若 LGB 与 XGB 单模型最优组合不同，可优先看业务指标（如 1 EUR FP）再手动微调。
已导出 → MODEL_FEATURES_V2_purgedcv.json


## 6.4a′ 换随机种子补测（purgedcv CV 不变）

> **注意：** `WalkForwardSplit` 折边界只由 `Time` 决定，**换种子不会改变 train/val 划分**。  
> 变的是：树模型初始化、折内早停分层切分、AE 子采样与 TF 权重 → 主要观察 **排名是否稳健**。

- `RUN_SEED`：改这一个整数即可（如 `123`、`7`）。
- `RERUN_AE=True`：连同 `ae_oof_error` 一起重算（更慢、更彻底）；`False` 只重跑 6.4a 组合矩阵（快）。
- **输出**：与 6.4a 相同——**完整 55 行** LGB / XGB 分表 + 双模型合并表 + 与种子 42 全量对照；并导出 `COMBO_MATRIX_purgedcv_seed{RUN_SEED}.json`。


In [14]:
# =============================================================================
# 6.4a′ 换随机种子补测 — 完整组合矩阵（镜像 6.4a 输出格式）
# =============================================================================
import json
from IPython.display import display

RUN_SEED = 123          # ← 改这里：42 / 123 / 7 / 2024 等
RERUN_AE = False        # True=重算 OOF AE + Family A 列；False=仅重跑组合 CV（快）

_required = ('df_fe', 'COMBO_SPECS', 'run_fe_combo_matrix', '_display_combo_table')
_missing = [n for n in _required if n not in globals()]
if _missing:
    raise RuntimeError(f'请先运行 §1–6.4a，缺少: {_missing}')

combo_lgb_seed42 = combo_lgb.copy() if 'combo_lgb' in globals() else None
combo_xgb_seed42 = combo_xgb.copy() if 'combo_xgb' in globals() else None

CV_RANDOM_STATE = RUN_SEED
AE_RANDOM_STATE = RUN_SEED

_orig_make_classifier = make_classifier

def make_classifier(model_name: str, y_train: pd.Series, params: dict | None = None):
    params = dict(params or {})
    spw = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
    if model_name == 'LightGBM':
        defaults = dict(
            n_estimators=MAX_BOOST_ROUNDS, learning_rate=0.05, max_depth=6, num_leaves=31,
            min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1,
            class_weight='balanced', random_state=RUN_SEED, verbose=-1,
        )
        defaults.update(params)
        return lgb.LGBMClassifier(**defaults)
    defaults = dict(
        n_estimators=MAX_BOOST_ROUNDS, learning_rate=0.05, max_depth=6,
        min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, scale_pos_weight=spw,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        random_state=RUN_SEED, eval_metric='logloss', verbosity=0,
    )
    defaults.update(params)
    defaults['early_stopping_rounds'] = EARLY_STOPPING_ROUNDS
    return xgb.XGBClassifier(**defaults)

try:
    if RERUN_AE:
        print(f'[seed={RUN_SEED}] 重算 OOF AE + 刷新 Family A 列…')
        TOP_V = pick_top_v_features(df_model, BASE_FEATURES, k=TOP_V_K, model_name='LightGBM')
        df_fe, CROSS_FAMILY_A = build_cross_features(
            df_model, TOP_V, gate_col='is_one_euro', prefix='one_euro',
        )
        df_fe['ae_oof_error'] = oof_ae_reconstruction_error(df_fe, feature_cols=AE_INPUT_COLS)
        df_fe['ae_oof_error_x_one_euro'] = df_fe['ae_oof_error'] * df_fe['is_one_euro'].astype(float)
        df_fe = bind_cv_data(sort_by_time(df_fe))

    print(f'6.4a′ 种子={RUN_SEED}（对照：6.4a 默认 CV_RANDOM_STATE=42）')
    print(f'共 {len(COMBO_SPECS)} 种组合 × {CV_N_SPLITS}-fold purgedcv WalkForwardSplit CV × 2 模型\n')

    print(f'=== 6.4a′ LightGBM 完整组合矩阵（seed={RUN_SEED}，按 AUC-PR_mean 降序）===')
    combo_lgb_alt = run_fe_combo_matrix(df_fe, COMBO_SPECS, model_name='LightGBM')
    _display_combo_table(combo_lgb_alt, '')

    print(f'\n=== 6.4a′ XGBoost 完整组合矩阵（seed={RUN_SEED}，按 AUC-PR_mean 降序）===')
    combo_xgb_alt = run_fe_combo_matrix(df_fe, COMBO_SPECS, model_name='XGBoost')
    _display_combo_table(combo_xgb_alt, '')

    print(f'\n--- LightGBM seed={RUN_SEED}：CV Δ>0 的前 3 名 ---')
    display(combo_lgb_alt.loc[combo_lgb_alt['Δ AUC-PR vs BASE'] > 0].head(3).drop(columns=['增量列']).round(4))

    print(f'--- XGBoost seed={RUN_SEED}：CV Δ>0 的前 3 名 ---')
    display(combo_xgb_alt.loc[combo_xgb_alt['Δ AUC-PR vs BASE'] > 0].head(3).drop(columns=['增量列']).round(4))

    combo_lgb_idx = combo_lgb_alt.set_index('特征组合')
    combo_xgb_idx = combo_xgb_alt.set_index('特征组合')
    _common_labels = combo_lgb_idx.index.intersection(combo_xgb_idx.index)
    combo_dual_alt = pd.DataFrame([
        {
            '特征组合': label,
            '增量列': combo_lgb_idx.loc[label, '增量列'],
            'Δ_LGB': float(combo_lgb_idx.loc[label, 'Δ AUC-PR vs BASE']),
            'Δ_XGB': float(combo_xgb_idx.loc[label, 'Δ AUC-PR vs BASE']),
            'Δ_mean': (
                float(combo_lgb_idx.loc[label, 'Δ AUC-PR vs BASE'])
                + float(combo_xgb_idx.loc[label, 'Δ AUC-PR vs BASE'])
            ) / 2,
            '双模型Δ均为正': (
                float(combo_lgb_idx.loc[label, 'Δ AUC-PR vs BASE']) > 0
                and float(combo_xgb_idx.loc[label, 'Δ AUC-PR vs BASE']) > 0
            ),
        }
        for label in _common_labels
    ]).sort_values('Δ_mean', ascending=False).reset_index(drop=True)

    print(f'\n=== 6.4a′ 双模型合并（seed={RUN_SEED}，全部 {len(combo_dual_alt)} 行，按 Δ_mean 降序）===')
    display(combo_dual_alt.drop(columns=['增量列']).round(4))

    _eligible_alt = combo_dual_alt.loc[combo_dual_alt['双模型Δ均为正']].sort_values('Δ_mean', ascending=False)
    print(f'\n=== seed={RUN_SEED} 双模型 Δ 均为正（共 {len(_eligible_alt)} 个）===')
    display(_eligible_alt.drop(columns=['增量列']).round(4) if not _eligible_alt.empty else _eligible_alt)

    def _seed_compare_full(ref_df: pd.DataFrame, alt_df: pd.DataFrame) -> pd.DataFrame:
        key = '特征组合'
        m = ref_df[[key, 'AUC-PR_mean', 'Δ AUC-PR vs BASE', 'FP', 'FN']].merge(
            alt_df[[key, 'AUC-PR_mean', 'Δ AUC-PR vs BASE', 'FP', 'FN']],
            on=key, suffixes=('_seed42', f'_seed{RUN_SEED}'),
        )
        m[f'ΔAP({RUN_SEED}−42)'] = m[f'AUC-PR_mean_seed{RUN_SEED}'] - m['AUC-PR_mean_seed42']
        m['Δ_diff'] = m[f'Δ AUC-PR vs BASE_seed{RUN_SEED}'] - m['Δ AUC-PR vs BASE_seed42']
        m['符号翻转'] = (m['Δ AUC-PR vs BASE_seed42'] * m[f'Δ AUC-PR vs BASE_seed{RUN_SEED}'] < 0)
        return m.sort_values(f'ΔAP({RUN_SEED}−42)', key=lambda s: s.abs(), ascending=False).reset_index(drop=True)

    if combo_lgb_seed42 is not None and combo_xgb_seed42 is not None:
        cmp_lgb = _seed_compare_full(combo_lgb_seed42, combo_lgb_alt)
        cmp_xgb = _seed_compare_full(combo_xgb_seed42, combo_xgb_alt)
        print(f'\n=== 种子 42 vs {RUN_SEED} 全量对照 — LightGBM（{len(cmp_lgb)} 行）===')
        display(cmp_lgb.round(4))
        print(f'\n=== 种子 42 vs {RUN_SEED} 全量对照 — XGBoost（{len(cmp_xgb)} 行）===')
        display(cmp_xgb.round(4))
        _flips = pd.concat([
            cmp_lgb.loc[cmp_lgb['符号翻转'], ['特征组合']].assign(模型='LightGBM'),
            cmp_xgb.loc[cmp_xgb['符号翻转'], ['特征组合']].assign(模型='XGBoost'),
        ], ignore_index=True)
        print(f'\n=== Δ 符号翻转（42 与 {RUN_SEED} 一正一负）共 {len(_flips)} 条 ===')
        display(_flips if not _flips.empty else pd.DataFrame(columns=['特征组合', '模型']))
    else:
        cmp_lgb = cmp_xgb = None
        print('\n（未找到 combo_lgb/combo_xgb，跳过与种子 42 全量对照）')

    export_seed_path = f'COMBO_MATRIX_purgedcv_seed{RUN_SEED}.json'

    def _df_records(df: pd.DataFrame) -> list:
        out = df.copy()
        if '增量列' in out.columns:
            out = out.drop(columns=['增量列'])
        return json.loads(out.round(6).to_json(orient='records', force_ascii=False))

    payload = {
        'cv_method': 'purgedcv.WalkForwardSplit',
        'cv_n_splits': CV_N_SPLITS,
        'cv_embargo': str(CV_EMBARGO),
        'run_seed': RUN_SEED,
        'ref_seed': 42,
        'rerun_ae': RERUN_AE,
        'n_combos': len(COMBO_SPECS),
        'combo_lgb': _df_records(combo_lgb_alt),
        'combo_xgb': _df_records(combo_xgb_alt),
        'combo_dual': _df_records(combo_dual_alt),
    }
    if cmp_lgb is not None:
        payload['seed_compare_lgb'] = _df_records(cmp_lgb)
        payload['seed_compare_xgb'] = _df_records(cmp_xgb)

    with open(export_seed_path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f'\n已导出完整矩阵 → {export_seed_path}')

finally:
    make_classifier = _orig_make_classifier


6.4a′ 种子=123（对照：6.4a 默认 CV_RANDOM_STATE=42）
共 55 种组合 × 5-fold purgedcv WalkForwardSplit CV × 2 模型

=== 6.4a′ LightGBM 完整组合矩阵（seed=123，按 AUC-PR_mean 降序）===
  [LightGBM] CV 1/55: 0. 基线（仅 BASE）
  [LightGBM] CV 2/55: Ed1. BASE + log1p_amount
  [LightGBM] CV 3/55: Ed2. BASE + hours_since_start
  [LightGBM] CV 4/55: Ed3. BASE + is_micro_testing
  [LightGBM] CV 5/55: Ed4. BASE + is_one_euro
  [LightGBM] CV 6/55: Ed5. BASE + is_amount_1_30
  [LightGBM] CV 7/55: Ed6. BASE + is_amount_75_110
  [LightGBM] CV 8/55: Ed_all. BASE + 全部 EDA（6 列）
  [LightGBM] CV 9/55: Ed_noBands. BASE + EDA 去掉难样本金额带
  [LightGBM] CV 10/55: Ed_bands. BASE + 两档难样本金额带
  [LightGBM] CV 11/55: Ed_noHours. BASE + EDA 再去掉 hours
  [LightGBM] CV 12/55: Ed_noHoursLog. BASE + EDA 再去掉 log1p
  [LightGBM] CV 13/55: Ed_cur. BASE + hours + is_one_euro
  [LightGBM] CV 14/55: AE. BASE + ae_oof_error
  [LightGBM] CV 15/55: AE+gate. BASE + ae_oof_error + ae×1EUR
  [LightGBM] CV 16/55: Ed_all+AE. 全部 EDA + ae_oof_error
  [LightGBM] CV 17/55: 

,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE
0,A3+AE. BASE + ae + one_euro_V12（V12）,2,32,"ae_oof_error, one_euro_V12",0.7899,0.0283,0.6525,0.8474,0.5305,47,231,0.0096
1,Ed_noHoursLog. BASE + EDA 再去掉 log1p,2,32,"is_micro_testing, is_one_euro",0.7882,0.0295,0.6491,0.8464,0.5264,47,233,0.0078
2,Ed_cur+A_top2. curated EDA + Family A Top-2,4,34,"hours_since_start, is_one_euro, one_euro_V14, ...",0.7875,0.0237,0.6600,0.8623,0.5346,42,229,0.0072
3,A4. BASE + one_euro_V7（V7）,1,31,one_euro_V7,0.7873,0.0296,0.6576,0.8439,0.5386,49,227,0.0069
4,AE+is_amount_1_30. ae_oof_error + is_amount_1_30,2,32,"ae_oof_error, is_amount_1_30",0.7871,0.0296,0.6583,0.8567,0.5346,44,229,0.0068
5,AE+log1p_amount. ae_oof_error + log1p_amount,2,32,"ae_oof_error, log1p_amount",0.7866,0.0283,0.6600,0.8471,0.5407,48,226,0.0062
6,Ed_all+A_top3. 全部 EDA + Family A Top-3,9,39,"log1p_amount, hours_since_start, is_micro_test...",0.7861,0.0209,0.6667,0.8693,0.5407,40,226,0.0058
7,Ed6. BASE + is_amount_75_110,1,31,is_amount_75_110,0.7858,0.0288,0.6600,0.8571,0.5366,44,228,0.0055
8,Ed4. BASE + is_one_euro,1,31,is_one_euro,0.7857,0.0312,0.6533,0.8553,0.5285,44,232,0.0053
9,AE+hours_since_start. ae_oof_error + hours_sin...,2,32,"ae_oof_error, hours_since_start",0.7856,0.0325,0.6599,0.8675,0.5325,40,230,0.0053



=== 6.4a′ XGBoost 完整组合矩阵（seed=123，按 AUC-PR_mean 降序）===
  [XGBoost] CV 1/55: 0. 基线（仅 BASE）
  [XGBoost] CV 2/55: Ed1. BASE + log1p_amount
  [XGBoost] CV 3/55: Ed2. BASE + hours_since_start
  [XGBoost] CV 4/55: Ed3. BASE + is_micro_testing
  [XGBoost] CV 5/55: Ed4. BASE + is_one_euro
  [XGBoost] CV 6/55: Ed5. BASE + is_amount_1_30
  [XGBoost] CV 7/55: Ed6. BASE + is_amount_75_110
  [XGBoost] CV 8/55: Ed_all. BASE + 全部 EDA（6 列）
  [XGBoost] CV 9/55: Ed_noBands. BASE + EDA 去掉难样本金额带
  [XGBoost] CV 10/55: Ed_bands. BASE + 两档难样本金额带
  [XGBoost] CV 11/55: Ed_noHours. BASE + EDA 再去掉 hours
  [XGBoost] CV 12/55: Ed_noHoursLog. BASE + EDA 再去掉 log1p
  [XGBoost] CV 13/55: Ed_cur. BASE + hours + is_one_euro
  [XGBoost] CV 14/55: AE. BASE + ae_oof_error
  [XGBoost] CV 15/55: AE+gate. BASE + ae_oof_error + ae×1EUR
  [XGBoost] CV 16/55: Ed_all+AE. 全部 EDA + ae_oof_error
  [XGBoost] CV 17/55: Ed_cur+AE. hours+is_one_euro + ae_oof_error
  [XGBoost] CV 18/55: Ed_noHoursLog+AE. EDA(无金额带/hours/log1p) + AE
  [XG

,特征组合,增量列数,总特征数,增量摘要,AUC-PR_mean,AUC-PR_std,F1@0.5,Precision@0.5,Recall@0.5,FP,FN,Δ AUC-PR vs BASE
0,A_top2+AE. ae + Family A Top-2,3,33,"ae_oof_error, one_euro_V14, one_euro_V4",0.7835,0.0283,0.6600,0.8623,0.5346,42,229,0.0052
1,Ed_noBands. BASE + EDA 去掉难样本金额带,4,34,"log1p_amount, hours_since_start, is_micro_test...",0.7834,0.0308,0.6551,0.8408,0.5366,50,228,0.0050
2,A4. BASE + one_euro_V7（V7）,1,31,one_euro_V7,0.7812,0.0331,0.6517,0.8397,0.5325,50,230,0.0028
3,Ed_noHours. BASE + EDA 再去掉 hours,3,33,"log1p_amount, is_micro_testing, is_one_euro",0.7811,0.0338,0.6550,0.8557,0.5305,44,231,0.0027
4,A1. BASE + one_euro_V14（V14）,1,31,one_euro_V14,0.7810,0.0318,0.6484,0.8339,0.5305,52,231,0.0026
5,Ed6. BASE + is_amount_75_110,1,31,is_amount_75_110,0.7808,0.0337,0.6476,0.8360,0.5285,51,232,0.0024
6,A2. BASE + one_euro_V4（V4）,1,31,one_euro_V4,0.7807,0.0329,0.6484,0.8387,0.5285,50,232,0.0024
7,Ed_noHoursLog+AE. EDA(无金额带/hours/log1p) + AE,3,33,"is_micro_testing, is_one_euro, ae_oof_error",0.7800,0.0337,0.6508,0.8520,0.5264,45,233,0.0016
8,AE+is_one_euro. ae_oof_error + is_one_euro,2,32,"ae_oof_error, is_one_euro",0.7800,0.0347,0.6558,0.8586,0.5305,43,231,0.0016
9,FULL. BASE + 全部人为特征（EDA+AE+FamilyA+gate）,14,44,"log1p_amount, hours_since_start, is_micro_test...",0.7797,0.0396,0.6616,0.8788,0.5305,36,231,0.0013



--- LightGBM seed=123：CV Δ>0 的前 3 名 ---


...


=== 种子 42 vs 123 全量对照 — XGBoost（55 行）===


,特征组合,AUC-PR_mean_seed42,Δ AUC-PR vs BASE_seed42,FP_seed42,FN_seed42,AUC-PR_mean_seed123,Δ AUC-PR vs BASE_seed123,FP_seed123,FN_seed123,ΔAP(123−42),Δ_diff,符号翻转
0,Ed_noBands. BASE + EDA 去掉难样本金额带,0.7745,-0.0042,47,227,0.7834,0.0050,50,228,0.0089,0.0092,True
1,A5. BASE + one_euro_V26（V26）,0.7820,0.0033,41,231,0.7750,-0.0034,51,234,-0.0070,-0.0067,True
2,Ed_all. BASE + 全部 EDA（6 列）,0.7812,0.0025,41,231,0.7742,-0.0042,44,231,-0.0070,-0.0067,True
3,Ed4. BASE + is_one_euro,0.7829,0.0042,46,229,0.7762,-0.0022,51,230,-0.0067,-0.0064,True
4,A3+AE. BASE + ae + one_euro_V12（V12）,0.7803,0.0016,40,230,0.7740,-0.0044,41,231,-0.0063,-0.0061,True
5,FULL. BASE + 全部人为特征（EDA+AE+FamilyA+gate）,0.7734,-0.0053,40,232,0.7797,0.0013,36,231,0.0063,0.0066,True
6,A4+AE. BASE + ae + one_euro_V7（V7）,0.7802,0.0015,39,228,0.7748,-0.0035,43,230,-0.0053,-0.0050,True
7,AE+hours_since_start. ae_oof_error + hours_sin...,0.7803,0.0017,39,228,0.7751,-0.0033,44,230,-0.0052,-0.0049,True
8,AE. BASE + ae_oof_error,0.7806,0.0019,40,228,0.7755,-0.0029,51,229,-0.0051,-0.0048,True
9,A1+AE. BASE + ae + one_euro_V14（V14）,0.7805,0.0019,39,229,0.7757,-0.0027,41,230,-0.0049,-0.0046,True



=== Δ 符号翻转（42 与 123 一正一负）共 62 条 ===


,特征组合,模型
0,A3+AE. BASE + ae + one_euro_V12（V12）,LightGBM
1,AE+is_amount_1_30. ae_oof_error + is_amount_1_30,LightGBM
2,AE+log1p_amount. ae_oof_error + log1p_amount,LightGBM
3,A6. BASE + one_euro_V10（V10）,LightGBM
4,Ed_cur+A_top2. curated EDA + Family A Top-2,LightGBM
...,...,...
57,Ed_noHoursLog. BASE + EDA 再去掉 log1p,XGBoost
58,Ed_all+AE+A_top3. 全部 EDA + ae + Family A Top-3,XGBoost
59,A6+AE. BASE + ae + one_euro_V10（V10）,XGBoost
60,Ed1. BASE + log1p_amount,XGBoost



已导出完整矩阵 → COMBO_MATRIX_purgedcv_seed123.json


都用A2较稳（one_euro_V4）